# 第 1 周末练习 —— 代码解释器（OpenAI + Ollama）

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一段代码 / 一个技术问题
- **输出**：清晰、信息量够用的解释
- **对比**：同一道题分别用云端 GPT 与本地 Llama，并体验**流式**与**非流式**两种写法

这是你在课程期间自己也能天天用的工具：看不懂的代码丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | `system_prompt` 定角色，`question` 放代码 |
| 流式输出 `stream=True` | `display_stream` + `update_display` 边收边刷新 |
| OpenAI 云端模型 | `MODEL_GPT = 'gpt-4o-mini'` |
| Ollama（OpenAI 兼容） | `base_url=http://localhost:11434/v1` + `MODEL_LLAMA` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；本地部分需已安装并启动 Ollama，并 `ollama pull llama3.2`
3. 在「提问」单元格改写 `question`，再分别跑 GPT / Llama 的流式与非流式格，对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 加载 .env；override=True 表示：即使进程里已有同名环境变量，也用 .env 里的值覆盖
load_dotenv(override = True)
# 从环境变量取出 OpenAI 密钥（本格读了但后面 OpenAI() 默认也会自己读同名变量）
api_key = os.getenv("OPENAI_API_KEY")
# 从 openai 导入 OpenAI 客户端类：调用云端（或兼容端点的）Chat Completions API
from openai import OpenAI
# 从 IPython.display 导入展示工具：Markdown 渲染、display 显示、update_display 原地刷新（流式常用）
from IPython.display import Markdown, display, update_display


In [ ]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境准备占位格 ==========
# 原作者在此留了「set up environment」提示；密钥加载已在上面的导入格完成
# 若你想在这里额外检查 api_key 是否为空，可自行加判断（本练习保持原逻辑不动）


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# system prompt：定模型角色与回答风格（发给模型的指令，保留英文，改译会改变行为）
system_prompt = "You are a code analyser and explainer. You will be given a piece of code and you will explain what it does and why. Try to cover all the aspects of the code. Use examples if necessary. Be concise but informative."

# user 侧的具体问题：三引号字符串里放待解释的代码；可整段改写成你自己的问题
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 过渡说明：接下来用 gpt-4o-mini，并演示流式回答 ==========
# 真正的 API 调用在后面「使用 OpenAI / 使用流式输出」单元格


# 使用 OpenAI

下面先创建云端客户端，再分别演示**流式**与**非流式**两种调用方式。


In [ ]:
# ========== 创建 OpenAI 客户端 ==========

# 不显式传 api_key 时，SDK 会自动从环境变量 OPENAI_API_KEY 读取
openai = OpenAI()


### 使用流式输出

`stream=True`：模型一边生成一边返回增量 `delta`；用 `update_display` 在同一个输出位刷新，体验像打字机。


In [ ]:
# ========== OpenAI 流式：发起请求 + 边收边刷新 Markdown ==========

# 定义函数：把 question 发给 GPT，返回可迭代的 stream 对象（还没把文本拼完）
def get_code_exp(question):
    # chat.completions.create：Chat Completions 主入口
    stream = openai.chat.completions.create(
        # 使用上面定义的云端模型常量
        model=MODEL_GPT,
        # messages：system 定角色，user 放具体代码问题
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
          ],
        # stream=True：不要等整段生成完，而是持续返回增量 chunk
        stream=True
    )  
    # 把 stream 交给调用方，由其决定怎么展示
    return stream  

# 定义函数：消费 stream，把每一块拼进 response，并用 update_display 原地更新
def display_stream(stream):
    # response：累积目前已收到的全部文本
    response = ""
    # display_id=True：拿到可更新的显示句柄，后续同一 id 刷新而不是新开一块输出
    display_handle = display(Markdown(""), display_id=True)
    # 逐块遍历流式事件
    for chunk in stream:
        # delta.content 可能为 None（例如结束标记）；用 or '' 避免把 None 拼进字符串
        response += chunk.choices[0].delta.content or ''
        # 用累积后的完整 Markdown 刷新同一显示位
        update_display(Markdown(response), display_id=display_handle.display_id)

# 用当前 question 发起流式请求并展示
display_stream(get_code_exp(question))


### 不使用流式输出

一次等完整回答返回，再 `display(Markdown(...))` 整段渲染。写法更简单，但要等模型全部生成完。


In [ ]:
# ========== OpenAI 非流式：等整段返回再展示 ==========

# 定义函数：同步调用 Chat Completions，直接返回完整字符串
def get_code_exp(question):
    # 注意：这里没有 stream=True，SDK 会等到整段生成完再返回
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
            ]
    )
    # 回复正文在 choices[0].message.content
    result = response.choices[0].message.content
    return result

# 调用函数并把结果按 Markdown 渲染到笔记本
display(Markdown(get_code_exp(question)))


# 使用 Llama（Ollama）

下面通过 **OpenAI 兼容端点** 调本地 Ollama：同一套 `chat.completions` 写法，只改 `base_url` / `api_key` / `model`。


In [ ]:
# ========== 拉取本地模型（shell 魔法） ==========

# Jupyter 的 ! 前缀：在 shell 里执行 ollama pull，把 llama3.2 下载到本机
!ollama pull llama3.2


In [ ]:
# ========== 指向本地 Ollama 的 OpenAI 兼容客户端 ==========

# Ollama 提供的 OpenAI 兼容基址（注意末尾 /v1）
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# api_key 对 Ollama 来说通常只是占位（非空即可）；真正区分后端的是 base_url
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


In [ ]:
# ========== 快速检查：在输出里看一眼客户端对象是否创建成功 ==========

# 直接写表达式：笔记本会显示该对象的 repr，便于确认没有报错
ollama


### 不使用流式输出

与上面 GPT 非流式格对称：同一函数形状，客户端换成 `ollama`，模型换成 `MODEL_LLAMA`。


In [ ]:
# ========== Llama 非流式：本地模型一次返回完整解释 ==========

# 定义函数：通过 Ollama 兼容端点同步调用
def get_code_exp(question):
    # 客户端是 ollama，模型是本地 MODEL_LLAMA；messages 结构与 GPT 完全相同
    response = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
            ]
    )
    # 取出助手回复正文
    result = response.choices[0].message.content
    return result

# 渲染 Markdown 结果
display(Markdown(get_code_exp(question)))


### 使用流式输出

本地 Llama 同样可以 `stream=True`；展示逻辑复用与 GPT 流式格相同的 `update_display` 模式。


In [ ]:
# ========== Llama 流式：本地模型边生成边刷新 ==========

# 发起流式请求的函数（与 GPT 流式版结构对称，只换客户端与模型名）
def get_code_exp(question):
    stream = ollama.chat.completions.create(
        # 本地模型名必须和 ollama list / pull 的一致
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
          ],
        # 打开流式
        stream=True
    )  
    return stream  

# 消费 stream 并原地更新 Markdown 显示
def display_stream(stream):
    # 累积缓冲区
    response = ""
    # 先放一个空的 Markdown 占位，拿到 display_id
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        # 拼接增量；or '' 防止 None
        response += chunk.choices[0].delta.content or ''
        # 刷新同一显示位
        update_display(Markdown(response), display_id=display_handle.display_id)

# 对当前 question 跑一遍 Llama 流式解释
display_stream(get_code_exp(question))


In [ ]:
# （空单元格：原笔记本保留；可在此自行追加对比实验）
